# 05 · Fine-tuning, LoRA/PEFT & Efficient Training

| | |
|---|---|
| **Author** | Chandan C |
| **GitHub** | [github.com/chandanc5525](https://github.com/chandanc5525) |
| **Course** | Agentic AI BootCamp · Module 02 – Advanced PyTorch |
| **Repository** | [AgenticAI_BootCamp-/02_advanced_pytorch](https://github.com/chandanc5525/AgenticAI_BootCamp-/tree/main/02_advanced_pytorch) |
| **Level** | Advanced |
| **Prerequisites** | Notebooks 01-04 |

---

### What you will learn
- Transfer learning: freezing, discriminative learning rates
- Forward / backward hooks & feature extraction
- LoRA implemented from scratch (the core of PEFT / QLoRA)
- Mixed precision (`autocast`, `GradScaler`), gradient accumulation, gradient checkpointing
- `torch.compile`, dynamic quantisation, profiling
- Distributed training concepts (DDP / FSDP) and the Hugging Face bridge

> Every cell runs on a plain CPU. If a GPU (CUDA / Apple MPS) is present, the notebooks use it automatically.

## 0. Setup

In [ ]:
import math, time, copy, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0); random.seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, device)

def count(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return f"total {total:,} | trainable {train:,} ({100*train/total:.2f}%)"

## 1. Transfer learning in one picture
1. Start from a model **pre-trained** on a huge dataset.
2. **Freeze** the backbone (or most of it).
3. Replace / add a **task head** and train it.
4. Optionally **unfreeze** top layers with a *smaller* learning rate.

With torchvision you would write `torchvision.models.resnet18(weights="DEFAULT")` (needs internet). To stay offline we create a small "pre-trained" backbone ourselves by pre-training on one task and then transferring to another.

In [ ]:
class Backbone(nn.Module):
    def __init__(self, in_dim=20, hid=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Linear(hid, hid), nn.ReLU())
    def forward(self, x): return self.net(x)

class Classifier(nn.Module):
    def __init__(self, backbone, n_classes, hid=128):
        super().__init__(); self.backbone = backbone; self.head = nn.Linear(hid, n_classes)
    def forward(self, x): return self.head(self.backbone(x))

def make_task(n, n_classes, task_seed, data_seed):
    """All tasks share the same latent feature map W (this is what transfers);
    each task has its own label mapping V."""
    W = torch.randn(20, 8, generator=torch.Generator().manual_seed(123))
    V = torch.randn(8, n_classes, generator=torch.Generator().manual_seed(task_seed))
    X = torch.randn(n, 20, generator=torch.Generator().manual_seed(data_seed))
    y = (torch.tanh(X @ W) @ V).argmax(1)
    return X, y

def train(model, X, y, epochs=30, lr=1e-3, params=None, bs=64):
    dl = DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr)
    for _ in range(epochs):
        for xb, yb in dl:
            loss = F.cross_entropy(model(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
@torch.no_grad()
def acc(model, X, y): model.eval(); return (model(X.to(device)).argmax(1).cpu() == y).float().mean().item()

# "Pre-training" on a big source task
Xs, ys = make_task(5000, 6, task_seed=1, data_seed=11)
pre = Classifier(Backbone(), 6).to(device); train(pre, Xs, ys, epochs=15)
print("source acc:", round(acc(pre, Xs, ys), 3))

# Small downstream task (only 100 labelled samples!)
Xt, yt = make_task(60, 4, task_seed=2, data_seed=12); Xv, yv = make_task(1000, 4, task_seed=2, data_seed=13)

In [ ]:
# A) from scratch
scratch = Classifier(Backbone(), 4).to(device); train(scratch, Xt, yt, epochs=40)
# B) frozen backbone + new head
fe = Classifier(copy.deepcopy(pre.backbone), 4).to(device)
for p in fe.backbone.parameters(): p.requires_grad = False
train(fe, Xt, yt, epochs=40, params=fe.head.parameters())
# C) full fine-tune with discriminative LRs
ft = Classifier(copy.deepcopy(pre.backbone), 4).to(device)
opt_groups = [{"params": ft.backbone.parameters(), "lr": 1e-4}, {"params": ft.head.parameters(), "lr": 1e-3}]
dl = DataLoader(TensorDataset(Xt, yt), batch_size=32, shuffle=True)
opt = torch.optim.AdamW(opt_groups)
for _ in range(40):
    for xb, yb in dl:
        l = F.cross_entropy(ft(xb.to(device)), yb.to(device)); opt.zero_grad(); l.backward(); opt.step()

for name, m in [("from scratch", scratch), ("frozen backbone", fe), ("fine-tuned", ft)]:
    print(f"{name:16s} val-acc {acc(m, Xv, yv):.3f} | {count(m)}")

## 2. Hooks – inspecting and extracting intermediate features
Forward hooks give you any layer's output without modifying the model - the basis for feature extraction, interpretability and activation steering.

In [ ]:
feats = {}
def save_to(name):
    def hook(module, inp, out): feats[name] = out.detach()
    return hook

handles = [ft.backbone.net[1].register_forward_hook(save_to("relu1")),
           ft.backbone.net[3].register_forward_hook(save_to("relu2"))]
ft.eval(); _ = ft(Xv[:5].to(device))
print({k: tuple(v.shape) for k, v in feats.items()})
for h in handles: h.remove()                    # always remove hooks

# gradient hook example: monitor gradient norms per layer
grads = {}
for n, p in ft.named_parameters():
    p.register_hook(lambda g, n=n: grads.__setitem__(n, g.norm().item()))
ft.train(); F.cross_entropy(ft(Xv[:32].to(device)), yv[:32].to(device)).backward()
print({k: round(v, 3) for k, v in grads.items()})

## 3. LoRA from scratch
Fine-tuning every weight of a 7 B-parameter LLM is impractical. **LoRA** freezes the weight matrix `W` and learns a *low-rank update*:

$$W' = W + \frac{\alpha}{r} B A,\qquad A\in\mathbb R^{r\times d_{in}},\; B\in\mathbb R^{d_{out}\times r}$$

Only `A` and `B` (rank `r` ≪ d) are trained → often < 1 % of the parameters, and the adapter can be merged back into `W` for zero-latency inference.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))        # zero init => starts as identity change
        self.scale = alpha / r; self.drop = nn.Dropout(dropout); self.merged = False
    def forward(self, x):
        out = self.base(x)
        if not self.merged: out = out + (self.drop(x) @ self.A.T @ self.B.T) * self.scale
        return out
    @torch.no_grad()
    def merge(self):
        if not self.merged: self.base.weight += (self.B @ self.A) * self.scale; self.merged = True

def apply_lora(model, r=8, alpha=16, target=nn.Linear, skip=("head",)):
    for name, child in list(model.named_children()):
        if isinstance(child, target) and name not in skip:
            setattr(model, name, LoRALinear(child, r, alpha))
        else:
            apply_lora(child, r, alpha, target, skip)
    return model

base = Classifier(copy.deepcopy(pre.backbone), 4).to(device)
for p in base.parameters(): p.requires_grad = False
for p in base.head.parameters(): p.requires_grad = True              # new head stays trainable
lora = apply_lora(base, r=4, alpha=8).to(device)
print(lora); print(count(lora))

In [ ]:
train(lora, Xt, yt, epochs=40, lr=3e-3, params=[p for p in lora.parameters() if p.requires_grad])
print("LoRA val-acc:", round(acc(lora, Xv, yv), 3))

# merging adapters: identical outputs, no extra latency
lora.eval(); before = lora(Xv[:8].to(device))
for m in lora.modules():
    if isinstance(m, LoRALinear): m.merge()
print("merge preserves outputs:", torch.allclose(before, lora(Xv[:8].to(device)), atol=1e-5))

# only adapter weights need saving (KBs instead of GBs)
adapter_sd = {k: v for k, v in lora.state_dict().items() if k.endswith((".A", ".B"))}
print("adapter tensors:", list(adapter_sd))

**In practice** with Hugging Face:
```python
# pip install transformers peft bitsandbytes accelerate
from peft import LoraConfig, get_peft_model
cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")
model = get_peft_model(base_model, cfg); model.print_trainable_parameters()
```
**QLoRA** = LoRA on top of a 4-bit quantised frozen base model (`bitsandbytes`), enabling 7B-70B fine-tuning on one GPU.

## 4. Mixed precision
Run most math in `bfloat16` / `float16` for speed and memory, keep master weights & sensitive ops (softmax, loss) in `float32`.  
`float16` needs **loss scaling** (`GradScaler`); `bfloat16` has fp32's range and usually doesn't.

In [ ]:
model = Classifier(Backbone(), 4).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
use_cuda = device.type == "cuda"
amp_dtype = torch.float16 if use_cuda else torch.bfloat16
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

xb, yb = Xt[:64].to(device), yt[:64].to(device)
for step in range(20):
    with torch.autocast(device_type=device.type, dtype=amp_dtype):
        out = model(xb); loss = F.cross_entropy(out, yb)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
print("autocast output dtype:", out.dtype, "| loss:", round(loss.item(), 4))

## 5. Gradient accumulation - large effective batch on small memory

In [ ]:
model = Classifier(Backbone(), 4).to(device); opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
micro_bs, accum = 16, 4                                   # effective batch = 64
dl = DataLoader(TensorDataset(Xs, ys % 4), batch_size=micro_bs, shuffle=True)
opt.zero_grad()
for i, (xb, yb) in enumerate(dl):
    loss = F.cross_entropy(model(xb.to(device)), yb.to(device)) / accum   # scale loss!
    loss.backward()
    if (i + 1) % accum == 0:
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); opt.zero_grad()
    if i == 99: break
print("done, effective batch =", micro_bs * accum)

## 6. Gradient (activation) checkpointing
Trade compute for memory: don't store activations in the forward pass, recompute them during backward. Essential for long-context LLM training.

In [ ]:
from torch.utils.checkpoint import checkpoint

class DeepStack(nn.Module):
    def __init__(self, d=256, L=24, use_ckpt=False):
        super().__init__()
        self.layers = nn.ModuleList(nn.Sequential(nn.Linear(d, d), nn.GELU()) for _ in range(L)); self.ck = use_ckpt
    def forward(self, x):
        for l in self.layers:
            x = checkpoint(l, x, use_reentrant=False) if self.ck and self.training else l(x)
        return x

x = torch.randn(256, 256)
for ck in [False, True]:
    m = DeepStack(use_ckpt=ck).train(); x_ = x.clone().requires_grad_()
    t0 = time.perf_counter(); m(x_).sum().backward(); dt = time.perf_counter() - t0
    print(f"checkpointing={ck!s:5} | step time {dt*1000:.0f} ms")

## 7. `torch.compile`, quantisation and profiling

In [ ]:
m = Classifier(Backbone(), 4).eval(); xb = torch.randn(512, 20)
try:
    cm = torch.compile(m)                       # needs a C++ compiler (CPU) or Triton (GPU)
    with torch.no_grad(): cm(xb)
    print("torch.compile OK")
except Exception as e:
    print("torch.compile unavailable here ->", type(e).__name__)

# Dynamic int8 quantisation of Linear layers (CPU inference)
qm = torch.ao.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)
size = lambda mod: sum(p.numel() * p.element_size() for p in mod.state_dict().values() if p.dtype.is_floating_point)
print("fp32 weights (B):", size(m), "| quantised model output shape:", qm(xb).shape)
with torch.no_grad():
    print("max |fp32 - int8| =", (m(xb) - qm(xb)).abs().max().item())

In [ ]:
from torch.profiler import profile, ProfilerActivity
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with torch.no_grad(): m(torch.randn(1024, 20))
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=5))

## 8. Scaling out: DDP and FSDP (concepts)
| Technique | Idea | Use when |
|---|---|---|
| **DDP** | Replicate the model on each GPU, split the batch, all-reduce gradients | Model fits on one GPU |
| **FSDP / ZeRO** | Shard parameters, gradients and optimiser state across GPUs | Model too large for one GPU |
| **Tensor / pipeline parallel** | Split layers / matrices across devices | Very large LLMs |

Skeleton (run with `torchrun --nproc_per_node=4 train.py`):
```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
dist.init_process_group("nccl"); rank = dist.get_rank(); torch.cuda.set_device(rank)
model = DDP(model.to(rank), device_ids=[rank])
loader = DataLoader(ds, batch_size=32, sampler=DistributedSampler(ds))
for epoch in range(E):
    loader.sampler.set_epoch(epoch)       # reshuffle differently each epoch
    ...
```

## 9. Bridge to Hugging Face
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
lm  = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", torch_dtype=torch.bfloat16)
out = lm.generate(**tok("Explain LoRA in one line.", return_tensors="pt"), max_new_tokens=40)
print(tok.decode(out[0], skip_special_tokens=True))
```
Everything you built here (`nn.Module`, attention, LoRA, AMP, checkpointing) is precisely what those libraries use internally.

## Exercises
1. Sweep LoRA rank `r ∈ {1, 2, 4, 8, 16}` and plot accuracy vs. trainable parameters.
2. Apply LoRA to the attention `qkv` projection of the mini-GPT from notebook 04 and fine-tune on a new text.
3. Add *layer-wise learning-rate decay* to the fine-tuning recipe.
4. Implement a simple "unfreeze one block per epoch" (gradual unfreezing) schedule.
5. Measure memory (`torch.cuda.max_memory_allocated`) with and without checkpointing on a GPU.

---
### Summary & next step
Work through the exercises above before moving on. Next notebook: **06_genai_and_agentic_ai_with_pytorch.ipynb**.

*Notebook by **Chandan C** · [github.com/chandanc5525](https://github.com/chandanc5525) · Agentic AI BootCamp*